# Responses API test (non-streaming, streaming, system instructions, multi-turn chaining, lifecycle)

**QE Perspective:** We validate the end-to-end contract of the Responses API across key features: basic completion, streaming chunking, system instructions parameter handling, multi-turn chaining via `previous_response_id`, and response lifecycle (retrieval and deletion).

- **Non-streaming & Streaming:** One-shot basic responses and stream iteration.
- **System Instructions:** Custom instructions (`instructions` parameter) in non-streaming and streaming modes.
- **Multi-turn Chaining:** Multi-turn conversation context preservation using `previous_response_id`.
- **Response Lifecycle:** Retrieval (`client.responses.retrieve`) and deletion (`client.responses.delete`).

Config: `BASE_URL`, `INFERENCE_MODEL`. Run via pytest.


In [ ]:
# Setup
import os
from scripts.helpers import response_text
from openai import OpenAI


base_url = os.environ.get("BASE_URL")
model = os.environ.get("INFERENCE_MODEL")

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

openai_base_url = base_url.rstrip("/")
openai_base_url = (
    openai_base_url if openai_base_url.endswith("/v1") else openai_base_url + "/v1"
)
client = OpenAI(api_key="no-key-needed", base_url=openai_base_url)

In [ ]:
response = client.responses.create(
    model=model,
    input="What is the capital of France? Answer in one short sentence.",
    temperature=0,
)
assert response.status == "completed", (
    f"Expected status completed, got {response.status}"
)
assert response.output, "Expected at least one output item"
text = response_text(response)
assert "Paris" in text, f"Expected Paris in response, got: {text}"

## Streaming: iterate stream and capture full message

Validate chunking logic — stream yields chunks; concatenating them produces the full response.


In [ ]:
stream = client.responses.create(
    model=model,
    input="What is the capital of France? Answer in one short sentence.",
    stream=True,
)
chunks = []
full_text = ""
for chunk in stream:
    chunks.append(chunk)
    # Text deltas arrive as OutputTextDelta chunks with a .delta attribute
    delta = getattr(chunk, "delta", None)
    if isinstance(delta, str):
        full_text += delta
assert len(chunks) >= 1, "Expected at least one streamed chunk"
assert full_text.strip(), "Expected non-empty full message from stream"
assert "Paris" in full_text, (
    f"Expected Paris in streamed response, got: {full_text[:200]}"
)

## System Instructions

Validate that system instructions (`instructions` parameter) are accepted and applied in both non-streaming and streaming modes.


In [ ]:
# System instructions: non-streaming
sys_instructions = (
    "You are a helpful assistant. Keep your answer concise and state only the fact."
)
response_sys = client.responses.create(
    model=model,
    instructions=sys_instructions,
    input="What is 5 + 5?",
    temperature=0,
)
assert response_sys.status == "completed", (
    f"Expected status completed, got {response_sys.status}"
)
assert response_sys.output, "Expected at least one output item"
text_sys = response_text(response_sys)
assert "10" in text_sys, f"Expected 10 in response with instructions, got: {text_sys}"

# System instructions: streaming
stream_sys = client.responses.create(
    model=model,
    instructions=sys_instructions,
    input="What is 5 + 5?",
    stream=True,
    temperature=0,
)
sys_chunks = []
sys_full_text = ""
for chunk in stream_sys:
    sys_chunks.append(chunk)
    delta = getattr(chunk, "delta", None)
    if isinstance(delta, str):
        sys_full_text += delta
assert len(sys_chunks) >= 1, "Expected at least one streamed chunk with instructions"
assert "10" in sys_full_text, (
    f"Expected 10 in streamed response with instructions, got: {sys_full_text}"
)

## Multi-Turn Chaining (`previous_response_id`)

Validate multi-turn conversation context preservation using `previous_response_id`. Turn 1 sets a fact in memory; Turn 2 queries that fact referencing `previous_response_id`.


In [ ]:
# Turn 1: Establish context
turn1 = client.responses.create(
    model=model,
    input="Remember this project code: KUBERNETES-2026.",
    temperature=0,
)
assert turn1.status == "completed", (
    f"Turn 1 expected status completed, got {turn1.status}"
)
assert hasattr(turn1, "id") and turn1.id, "Expected valid response ID on turn1"

# Turn 2: Query using previous_response_id (non-streaming)
turn2 = client.responses.create(
    model=model,
    input="What is the project code I told you?",
    previous_response_id=turn1.id,
    temperature=0,
)
assert turn2.status == "completed", (
    f"Turn 2 expected status completed, got {turn2.status}"
)
text_turn2 = response_text(turn2)
assert "KUBERNETES-2026" in text_turn2, (
    f"Expected project code in multi-turn response, got: {text_turn2}"
)

# Turn 3: Query using previous_response_id (streaming)
stream_turn3 = client.responses.create(
    model=model,
    input="Repeat the project code one more time.",
    previous_response_id=turn2.id,
    stream=True,
    temperature=0,
)
turn3_full_text = ""
for chunk in stream_turn3:
    delta = getattr(chunk, "delta", None)
    if isinstance(delta, str):
        turn3_full_text += delta
assert "KUBERNETES-2026" in turn3_full_text, (
    f"Expected project code in streamed multi-turn response, got: {turn3_full_text}"
)

## Response Lifecycle: Retrieval and Deletion

Validate retrieving a response by ID (`client.responses.retrieve`) and deleting a response (`client.responses.delete`).


In [ ]:
from openai import APIError

# Create a response for retrieval and deletion
resp_lifecycle = client.responses.create(
    model=model,
    input="Ephemeral message for lifecycle test.",
    temperature=0,
)
assert resp_lifecycle.status == "completed", (
    f"Expected completed status, got {resp_lifecycle.status}"
)
resp_id = resp_lifecycle.id
assert resp_id, "Response must have a valid ID"

# Retrieve response by ID
retrieved = client.responses.retrieve(response_id=resp_id)
assert retrieved.id == resp_id, (
    f"Retrieved ID {retrieved.id} does not match created ID {resp_id}"
)
assert retrieved.status == "completed", (
    f"Expected completed status, got {retrieved.status}"
)

# Delete response by ID
delete_handled = False
try:
    client.responses.delete(response_id=resp_id)
    delete_handled = True
except APIError as e:
    delete_handled = e.status_code in (200, 204, 404)
assert delete_handled, "Delete operation must complete cleanly or return a valid status"

# Verify post-deletion retrieval behavior
post_delete_handled = False
try:
    post_del = client.responses.retrieve(response_id=resp_id)
    post_delete_handled = getattr(post_del, "status", None) in (
        "deleted",
        "not_found",
        "failed",
    )
except APIError as e:
    post_delete_handled = e.status_code in (404, 410, 400)
except Exception:
    post_delete_handled = True
assert post_delete_handled, (
    "Retrieving a deleted response must yield error or deleted status"
)